# Модуль 4. Ускоряем код: параллельные вычисления с `Parallel` и `delayed`

## Зачем этот модуль

Представьте, что у вас есть список из 100 задач. Каждая занимает 1 секунду. Если решать их по очереди (последовательно), уйдёт 100 секунд — почти две минуты. Но если дать по 25 задач четырём работникам, они закончат примерно за 25 секунд.

В Python есть встроенный модуль `multiprocessing`, который позволяет создавать несколько процессов. Но он требует писать много «связующего» кода: создавать пул процессов, разбивать данные, собирать результаты, обрабатывать ошибки.

`joblib` делает это проще. Он предоставляет инструменты `Parallel` и `delayed`, которые превращают обычный цикл `for` в параллельный — буквально добавлением пары строк.

В этом модуле мы разберём, как это работает, когда это ускоряет код, а когда — замедляет.

## 1. Почему Python не всегда ускоряется простым «распараллеливанием»

### Аналогия с кухней

Представьте ресторанную кухню. Есть два способа готовить быстрее:

1. **Нанять больше поваров** — каждый готовит своё блюдо на отдельной плите. Это **процессы**.
2. **Одному повару дать два ножа** — чтобы он резал одновременно двумя руками. Это **потоки**.

В Python **потоки** (threads) не ускоряют «тяжёлые» вычисления. Почему? Из-за **GIL**.

### Что такое GIL

**GIL** (Global Interpreter Lock, глобальная блокировка интерпретатора) — это механизм Python, который гарантирует: **в один момент времени только один поток выполняет Python-код**.

Аналогия: в комнате с доской для лекций есть один маркер. Даже если в комнате 20 студентов, маркер может держать только один. Остальные ждут в очереди.

Это значит, что если вы запустите 10 потоков для сложных вычислений — они не будут работать параллельно. Они будут быстро переключаться, но CPU загружен будет всё равно только на 100% одного ядра.

### Когда потоки (threads) всё-таки полезны

Потоки ускоряют **I/O-bound** задачи — те, где программа много ждёт:
- запросы к интернету;
- чтение/запись с диска;
- запросы к базе данных.

Пока один поток ждёт ответа от сервера, другой может работать. GIL в это время отпускается.

### Когда нужны процессы (processes)

Для **CPU-bound** задач — сложных вычислений (обучение моделей, обработка данных, математика) — нужны **процессы**. Каждый процесс получает свой собственный Python-интерпретатор со своим GIL. Поэтому 4 процесса могут загрузить 4 ядра CPU на 100%.

### Вывод перед кодом

| Тип задачи | Что использовать | Почему |
|------------|----------------|--------|
| Сетевые запросы, чтение файлов | Потоки (`threading`) | Пока ждём ответ, другие потоки работают |
| Математика, ML, обработка данных | Процессы (`multiprocessing`, `loky`) | Обход GIL, каждый процесс на своём ядре |
| Смешанные задачи | Процессы | Надёжнее для ML |

`joblib` по умолчанию использует процессы (`backend='loky'`), поэтому он подходит для ML «из коробки».

## 2. Синтаксис `Parallel` и `delayed`

### Базовый шаблон

In [ ]:
from joblib import Parallel, delayed

results = Parallel(n_jobs=-1)(
    delayed(some_function)(arg) for arg in items
)

Разберём по частям:

| Элемент | Что это |
|---------|---------|
| `Parallel(...)` | Конструктор, который создаёт «менеджера» параллельных задач |
| `n_jobs=-1` | Сколько процессов/потоков использовать. `-1` = все ядра CPU |
| `delayed(func)(arg)` | «Заморозка» вызова функции. Говорит: «вот функция и аргумент, выполни позже» |
| `(delayed(...)(...) for ... in ...)` | Генератор, который создаёт список «замороженных» вызовов |
| `Parallel(...)(...)` | Запускает все «замороженные» вызовы параллельно и собирает результаты |

### Простой пример: возведение в квадрат

In [ ]:
from joblib import Parallel, delayed
import time

def square(x):
    time.sleep(0.5)  # имитация работы
    return x ** 2

numbers = [1, 2, 3, 4]

# Последовательно
start = time.time()
result_seq = [square(n) for n in numbers]
print(f"Последовательно: {time.time() - start:.2f} сек, результат: {result_seq}")

# Параллельно
start = time.time()
result_par = Parallel(n_jobs=-1)(
    delayed(square)(n) for n in numbers
)
print(f"Параллельно: {time.time() - start:.2f} сек, результат: {result_par}")

**Типичный результат на 4-ядерном CPU:**

In [ ]:
Последовательно: 2.01 сек, результат: [1, 4, 9, 16]
Параллельно: 0.55 сек, результат: [1, 4, 9, 16]

4 задачи по 0.5 секунды выполнились почти за 0.5 секунды вместо 2 секунд — ускорение почти в 4 раза.

### Почему `delayed`?

`delayed` — это «обещание выполнить функцию позже». Он не вызывает функцию сразу, а создаёт объект с инструкцией: «вызови `square` с аргументом `n`». `Parallel` потом берёт эти инструкции и раздаёт процессам.

Без `delayed` Python попытался бы вызвать функцию сразу, ещё до параллельного запуска.

## 3. Параметры `Parallel`

### `n_jobs`: сколько «рук» работает

| Значение | Смысл |
|----------|-------|
| `1` | Без параллелизма (полезно для отладки) |
| `2`, `4`, `8` | Ровно столько процессов |
| `-1` | Все доступные ядра CPU |
| `-2` | Все ядра, кроме одного (оставить ресурсы системе) |

In [ ]:
# Использовать 2 процесса
Parallel(n_jobs=2)(...)

# Оставить одно ядро свободным
Parallel(n_jobs=-2)(...)

### `backend`: как именно распараллеливать

| Бэкенд | Что делает | Когда использовать |
|--------|-----------|-------------------|
| `'loky'` (по умолчанию) | Создаёт отдельные процессы, безопасно и удобно | **По умолчанию для ML**. Работает везде, включая Jupyter |
| `'threading'` | Создаёт потоки внутри одного процесса | Только для I/O-bound задач (запросы, чтение файлов) |
| `'multiprocessing'` | Использует стандартный `multiprocessing` | Если `loky` не справляется (редко) |

In [ ]:
# Явно указать потоки (для сетевых запросов)
Parallel(n_jobs=4, backend='threading')(
    delayed(fetch_url)(url) for url in urls
)

### `verbose`: видеть прогресс

| Значение | Что показывает |
|----------|---------------|
| `0` | Ничего |
| `1`–`5` | Общая информация |
| `10` | Подробный прогресс: сколько задач выполнено, сколько осталось |
| `50` | Максимум деталей |

In [ ]:
Parallel(n_jobs=-1, verbose=10)(
    delayed(train_model)(params) for params in param_list
)

Вывод будет примерно таким:

In [ ]:
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 tasks      | elapsed:    5.2s
[Parallel(n_jobs=4)]: Done   6 tasks      | elapsed:   10.1s

## 4. Когда `Parallel` ускоряет, а когда замедляет

### Параллелизм имеет цену

Создание процесса — это не бесплатно. Операционная система должна:
- выделить память;
- скопировать туда часть данных;
- запустить новый Python-интерпретатор;
- передать аргументы;
- собрать результаты обратно.

Если задача выполняется за 1 миллисекунду — накладные расходы на создание процесса (10–100 мс) сделают параллельный вариант **медленнее**.

### Правило: минимальное время задачи

Используйте `Parallel`, если **одна итерация цикла занимает больше 100 мс** (0.1 секунды). Если меньше — последовательный цикл может оказаться быстрее.

### Правило: размер данных

Если вы передаёте в каждый процесс огромный массив данных (например, датасет на 5 ГБ), `joblib` будет тратить время на копирование этих данных в каждый процесс. В таких случаях лучше:
- использовать `threading` (если задача I/O-bound);
- или загружать данные внутри функции, а не передавать как аргумент.

### Правило: порядок результатов

`Parallel` сохраняет порядок результатов — они придут в том же порядке, в каком были задачи. Но сами задачи выполняются в произвольном порядке.

In [ ]:
results = Parallel(n_jobs=-1)(
    delayed(square)(n) for n in [5, 3, 8, 1]
)
# results всегда [25, 9, 64, 1], даже если 8 посчиталась раньше 5

## 5. ML-сценарии

### Сценарий A: Ручной перебор гиперпараметров

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
from joblib import Parallel, delayed

iris = load_iris()
X, y = iris.data, iris.target

def evaluate_params(n_estimators, max_depth):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    scores = cross_val_score(model, X, y, cv=5)
    return {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'mean_score': scores.mean()
    }

# Сетка гиперпараметров
param_grid = [
    (50, 5), (50, 10), (50, None),
    (100, 5), (100, 10), (100, None),
    (200, 5), (200, 10), (200, None),
]

# Параллельный перебор
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(evaluate_params)(n, d) for n, d in param_grid
)

# Лучший результат
best = max(results, key=lambda x: x['mean_score'])
print("Лучшие параметры:", best)

Каждая комбинация гиперпараметров обучает модель независимо — идеально для параллелизма.

### Сценарий B: Параллельный feature engineering

In [ ]:
import pandas as pd
from joblib import Parallel, delayed

def create_feature(df, feature_name):
    """Создание одного сложного признака"""
    if feature_name == 'income_per_age':
        return df['income'] / df['age']
    elif feature_name == 'spending_score':
        return df['spending'] / df['income'].clip(lower=1)
    # ... и так далее

df = pd.read_csv('customers.csv')
features_to_create = ['income_per_age', 'spending_score', 'loyalty_index', 'recency']

# Создаём признаки параллельно
new_features = Parallel(n_jobs=-1)(
    delayed(create_feature)(df, name) for name in features_to_create
)

for name, series in zip(features_to_create, new_features):
    df[name] = series

### Сценарий C: Параллельное предсказание на чанках

In [ ]:
import numpy as np
from joblib import Parallel, delayed

def predict_chunk(model, chunk):
    return model.predict(chunk)

# Большой массив данных
X_large = np.random.rand(1_000_000, 10)  # миллион строк
chunk_size = 10_000
chunks = [X_large[i:i+chunk_size] for i in range(0, len(X_large), chunk_size)]

# Параллельный batch inference
predictions = Parallel(n_jobs=-1, verbose=5)(
    delayed(predict_chunk)(model, chunk) for chunk in chunks
)

# Собираем обратно
all_predictions = np.concatenate(predictions)

## 6. Backend-сценарии

### Сценарий D: Batch inference в API

In [ ]:
from fastapi import FastAPI
from joblib import Parallel, delayed
import numpy as np

app = FastAPI()

@app.post("/batch_predict")
def batch_predict(data: list[list[float]]):
    X = np.array(data)
    
    # Если много данных — разбиваем на чанки и параллелим
    if len(X) > 1000:
        chunk_size = 500
        chunks = [X[i:i+chunk_size] for i in range(0, len(X), chunk_size)]
        
        preds = Parallel(n_jobs=4)(
            delayed(lambda c: model.predict(c))(chunk) for chunk in chunks
        )
        return {"predictions": np.concatenate(preds).tolist()}
    
    # Мало данных — проще последовательно
    return {"predictions": model.predict(X).tolist()}

> **Важно:** внутри FastAPI/Flask не стоит создавать процессы на каждый запрос — это дорого. `Parallel` имеет смысл только для **batch**-запросов с большим количеством данных. Для одиночных запросов используйте просто `model.predict()`.

### Сценарий E: Параллельная загрузка данных из нескольких источников

In [ ]:
import requests
from joblib import Parallel, delayed

urls = [
    "https://api.example.com/data/2024-01",
    "https://api.example.com/data/2024-02",
    "https://api.example.com/data/2024-03",
]

def fetch_data(url):
    return requests.get(url, timeout=30).json()

# Потоки, потому что задача I/O-bound (ждём сеть)
datasets = Parallel(n_jobs=3, backend='threading')(
    delayed(fetch_data)(url) for url in urls
)

## 7. Подводные камни и типичные ошибки

### Ошибка 1: Передача лямбда-функций

In [ ]:
# НЕ РАБОТАЕТ — лямбда не сериализуется для передачи в процесс
results = Parallel(n_jobs=-1)(
    delayed(lambda x: x**2)(n) for n in range(10)
)

**Решение:** используйте обычные именованные функции.

In [ ]:
def square(x):
    return x ** 2

results = Parallel(n_jobs=-1)(
    delayed(square)(n) for n in range(10)
)

### Ошибка 2: Передача огромных объектов как аргументов

In [ ]:
# ПЛОХО: копирование 2 ГБ данных в каждый процесс
big_df = pd.read_csv('huge_file.csv')  # 2 ГБ

results = Parallel(n_jobs=8)(
    delayed(process)(big_df, i) for i in range(100)
)  # Медленно! Каждый процесс получит копию big_df

**Решение:** либо загружайте данные внутри функции, либо используйте `threading`, либо разбивайте задачу иначе.

### Ошибка 3: Использование `Parallel` внутри `Parallel`

Вложенные `Parallel` вызовы могут привести к deadlock (взаимной блокировке) или исчерпанию ресурсов.

### Ошибка 4: Забыли импортировать библиотеки внутри функции

Когда `joblib` создаёт новый процесс, он копирует только данные, а не всё состояние программы. Если функция использует библиотеку, импортированную только в основном скрипте — в процессе её может не быть.

**Решение:** импортируйте всё необходимое внутри функции или на уровне модуля.

In [ ]:
def train_model(params):
    from sklearn.ensemble import RandomForestClassifier  # импорт внутри
    # ...

## 8. Практика: задания

### Задание 4.1: «Первый параллельный цикл»

1. Напишите функцию `heavy_computation(n)`, которая:
   - считает сумму квадратов чисел от 1 до `n` (`sum(i**2 for i in range(n))`);
   - перед вычислением делает `time.sleep(0.5)`.
2. Создайте список чисел `[100000, 200000, 300000, 400000]`.
3. Выполните последовательно и замерьте время.
4. Выполните через `Parallel(n_jobs=-1)` и замерьте время.
5. Вычислите ускорение (во сколько раз быстрее).

### Задание 4.2: «Перебор гиперпараметров»

1. Используйте датасет `load_iris()`.
2. Создайте список из 6 комбинаций параметров для `RandomForestClassifier`:
   - `n_estimators`: 10, 50, 100;
   - `max_depth`: 3, 5.
3. Напишите функцию `evaluate(params)`, которая:
   - создаёт модель с этими параметрами;
   - делает `cross_val_score` с `cv=3`;
   - возвращает среднюю точность.
4. Выполните перебор последовательно и параллельно. Сравните время.

### Задание 4.3: «Потоки vs Процессы»

1. Напишите функцию `fetch_fake(url)`, которая делает `time.sleep(1)` (имитация сетевого запроса).
2. Создайте список из 8 URL-заглушек.
3. Выполните через `Parallel` с `backend='threading'` и замерьте время.
4. Выполните через `Parallel` с `backend='loky'` и замерьте время.
5. Объясните, почему `threading` в данном случае работает не хуже (а иногда лучше) процессов.

### Задание 4.4: «Batch inference»

1. Создайте массив `numpy` размером `5000 × 10` случайных чисел.
2. Загрузите любую обученную модель (или создайте `RandomForest` на лету).
3. Разбейте массив на 10 чанков по 500 строк.
4. Сделайте предсказание на каждом чанке параллельно через `Parallel`.
5. Соберите результаты в один массив через `np.concatenate`.

## 9. Эталонное решение

### Решение 4.1

In [ ]:
from joblib import Parallel, delayed
import time

def heavy_computation(n):
    time.sleep(0.5)  # имитация работы
    return sum(i**2 for i in range(n))

numbers = [100_000, 200_000, 300_000, 400_000]

# Последовательно
start = time.time()
results_seq = [heavy_computation(n) for n in numbers]
time_seq = time.time() - start

# Параллельно
start = time.time()
results_par = Parallel(n_jobs=-1)(
    delayed(heavy_computation)(n) for n in numbers
)
time_par = time.time() - start

print(f"Последовательно: {time_seq:.2f} сек")
print(f"Параллельно: {time_par:.2f} сек")
print(f"Ускорение: {time_seq / time_par:.1f}x")
print(f"Результаты: {results_par}")

**Ожидаемый вывод (на 4-ядерном CPU):**

In [ ]:
Последовательно: 2.01 сек
Параллельно: 0.52 сек
Ускорение: 3.9x

### Решение 4.2

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from joblib import Parallel, delayed
import time

iris = load_iris()
X, y = iris.data, iris.target

def evaluate(params):
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X, y, cv=3)
    return {'params': params, 'score': scores.mean()}

param_grid = [
    {'n_estimators': 10, 'max_depth': 3},
    {'n_estimators': 10, 'max_depth': 5},
    {'n_estimators': 50, 'max_depth': 3},
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 3},
    {'n_estimators': 100, 'max_depth': 5},
]

# Последовательно
start = time.time()
results_seq = [evaluate(p) for p in param_grid]
print(f"Последовательно: {time.time() - start:.2f} сек")

# Параллельно
start = time.time()
results_par = Parallel(n_jobs=-1, verbose=10)(
    delayed(evaluate)(p) for p in param_grid
)
print(f"Параллельно: {time.time() - start:.2f} сек")

best = max(results_par, key=lambda x: x['score'])
print("Лучший результат:", best)

## 10. Вопросы для самопроверки

1. **Почему потоки (threads) в Python не ускоряют математические вычисления?**  
   *(Ответ: из-за GIL — глобальной блокировки интерпретатора. В один момент времени только один поток выполняет Python-код.)*

2. **Какой бэкенд в `joblib` используется по умолчанию и почему он подходит для ML?**  
   *(Ответ: `'loky'`. Он создаёт отдельные процессы, каждый со своим GIL, поэтому задачи реально выполняются параллельно на разных ядрах CPU.)*

3. **Что делает `delayed(func)(arg)`?**  
   *(Ответ: создаёт объект-инструкцию «вызвать func с аргументом arg», но не вызывает функцию сразу. `Parallel` потом выполняет эти инструкции параллельно.)*

4. **Когда `Parallel` может замедлить программу вместо ускорения?**  
   *(Ответ: когда одна задача выполняется очень быстро (меньше 100 мс) — накладные расходы на создание процессов превысят выигрыш. Или когда передаются огромные объекты как аргументы.)*

5. **Почему в `Parallel` нельзя передавать `lambda`-функции?**  
   *(Ответ: процессы не могут сериализовать (передать) лямбда-функции. Нужно использовать обычные именованные функции.)*

6. **В каком порядке приходят результаты из `Parallel`?**  
   *(Ответ: в том же порядке, в каком были переданы задачи, независимо от порядка их фактического выполнения.)*

## Итоги модуля

- **GIL** в Python не позволяет потокам ускорять CPU-bound задачи. Для вычислений нужны **процессы**.
- `Parallel(n_jobs=-1)` + `delayed(func)(arg)` — простой способ превратить цикл `for` в параллельный.
- По умолчанию `joblib` использует бэкенд `'loky'` (процессы), который идеален для ML.
- `backend='threading'` полезен только для I/O-bound задач (сеть, диск).
- Параллелизм имеет накладные расходы. Используйте его, когда одна задача длится **больше 100 мс**.
- В ML `Parallel` незаменим для: перебора гиперпараметров, кросс-валидации, batch inference, параллельного feature engineering.
- В backend `Parallel` применяется осторожно — только для batch-операций, не для каждого HTTP-запроса.

**В следующем модуле** мы научимся интегрировать `joblib` со scikit-learn — узнаем, как библиотека использует `joblib` под капотом, и научимся сохранять и загружать полные `Pipeline`.